In [204]:
from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings
import getpass
import os 
from langchain_nvidia_ai_endpoints import NVIDIAEmbeddings
from langchain_community.document_loaders import AsyncHtmlLoader # for lading the HTML content
from langchain_text_splitters import HTMLSectionSplitter # for splitting the HTML content
from dotenv import load_dotenv

load_dotenv()
from langchain_text_splitters import RecursiveCharacterTextSplitter


In [205]:
embeddings = NVIDIAEmbeddings(
        model="nvidia/nv-embedqa-e5-v5",
        api_key=os.getenv("NVIDIA_NV_API"),
)

In [206]:
cornwall_granular_collection = Chroma(
 collection_name="cornwall_granular",
 embedding_function=embeddings,
)

# reset the collection if it already exists
cornwall_granular_collection.reset_collection()

### Loading HTML content using AsyncHTML LOADER

In [207]:
destination_url = "https://en.wikivoyage.org/wiki/South_Cornwall"
html_loader = AsyncHtmlLoader(destination_url)
# add the documents to the collection
documents = html_loader.load()

Fetching pages: 100%|##########| 1/1 [00:00<00:00,  3.34it/s]


In [208]:
headers_to_split_on = [("h1", "Header 1"), ("h2", "Header 2")]
html_section_splitter = HTMLSectionSplitter(headers_to_split_on=headers_to_split_on)

In [209]:
def split_docs_into_granular_chunks(documents):
    all_chunks = []
    # Use small chunks to stay under embedding token limits
    fallback_splitter = RecursiveCharacterTextSplitter(
        chunk_size=450, # here the chunk size is the size of the context window
        chunk_overlap=50, # here the chunk overlap is the size of the overlap
    )

    for doc in documents:
        html_string = doc.page_content
        temp_chunks = html_section_splitter.split_text(html_string)
        all_chunks.extend(fallback_splitter.split_documents(temp_chunks))

    return all_chunks


In [210]:
# now split the documents into granular chunks
granular_chunks = split_docs_into_granular_chunks(documents)

#Now insert the granular chunks into the collection
cornwall_granular_collection.add_documents(granular_chunks)

['64ffe0b4-df14-4bec-bb88-f5e18154eaaf']

In [211]:
results = cornwall_granular_collection.similarity_search(
 query="abhilov",k=3)
for doc in results:
 print(doc.page_content)

Please respect our robot policy https://w.wiki/4wJS when crawling us. Contact bot-traffic@wikimedia.org if you need higher volumes. (30224bb)


### Embedding child chunks with ParentDocumentRetriever

In [212]:
from langchain_classic.retrievers import ParentDocumentRetriever
from langchain_classic.storage import InMemoryStore
from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings
from langchain_community.document_loaders import AsyncHtmlLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [213]:
parent_splitter = RecursiveCharacterTextSplitter(chunk_size=3000, chunk_overlap=50)
child_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)

In [214]:
child_chunks_collection = Chroma(
 collection_name="cornwall_granular",
 embedding_function=embeddings,
)

# reset the collection if it already exists
child_chunks_collection.reset_collection()

doc_store = InMemoryStore()

In [215]:
doc_store = InMemoryStore()

In [216]:
# ParentDocumentRetriever uses:
# - parent_splitter → splits documents into larger parent chunks
# - child_splitter → splits parent chunks into smaller child chunks for embedding
# - vectorstore → stores child chunks (used for similarity search)
# - docstore → stores full parent documents (used to return full context)
parent_doc_retriever = ParentDocumentRetriever(
    vectorstore=child_chunks_collection,
    docstore=doc_store,
    child_splitter=child_splitter,
    parent_splitter=parent_splitter
)

In [217]:
uk_destinations = [
 "Cornwall", "North_Cornwall", "South_Cornwall", "West_Cornwall",
 "Tintagel", "Bodmin", "Wadebridge", "Penzance", "Newquay",
 "St_Ives", "Port_Isaac", "Looe", "Polperro", "Porthleven",
 "East_Sussex", "Brighton", "Battle", "Hastings_(England)",
 "Rye_(England)", "Seaford", "Ashdown_Forest"
]

In [218]:
for destination_url in destination_url:
    if not destination_url.startswith(("http://", "https://")):
        print(f"Skipping invalid URL: {destination_url}")
        continue

    html_loader = AsyncHtmlLoader([destination_url])

    html_docs = html_loader.load()

    text_docs = html2text_transformer.transform_documents(html_docs)

    print(f'Ingesting {destination_url}')

    parent_doc_retriever.add_documents(text_docs)

Skipping invalid URL: h
Skipping invalid URL: t
Skipping invalid URL: t
Skipping invalid URL: p
Skipping invalid URL: s
Skipping invalid URL: :
Skipping invalid URL: /
Skipping invalid URL: /
Skipping invalid URL: e
Skipping invalid URL: n
Skipping invalid URL: .
Skipping invalid URL: w
Skipping invalid URL: i
Skipping invalid URL: k
Skipping invalid URL: i
Skipping invalid URL: v
Skipping invalid URL: o
Skipping invalid URL: y
Skipping invalid URL: a
Skipping invalid URL: g
Skipping invalid URL: e
Skipping invalid URL: .
Skipping invalid URL: o
Skipping invalid URL: r
Skipping invalid URL: g
Skipping invalid URL: /
Skipping invalid URL: w
Skipping invalid URL: i
Skipping invalid URL: k
Skipping invalid URL: i
Skipping invalid URL: /
Skipping invalid URL: S
Skipping invalid URL: o
Skipping invalid URL: u
Skipping invalid URL: t
Skipping invalid URL: h
Skipping invalid URL: _
Skipping invalid URL: C
Skipping invalid URL: o
Skipping invalid URL: r
Skipping invalid URL: n
Skipping invalid

In [219]:
list(doc_store.yield_keys()) 

[]

In [ ]:
# directly searching the data from the chunks 
child_docs_only = child_chunks_collection.similarity_search("Cornwall Ranger")

In [222]:
child_docs_only

[]